# Numbers Nobody Publishes: quant route

What this notebook does, in order:

1. Reads the client's list of licensees.
2. Looks each name up in an official register of the country written next to it, and keeps only the matches I am at least 75% sure about. Every match keeps the link it came from.
3. Builds a training table from the Romanian filings (one row per firm).
4. Trains two models: one for the average number of employees, one for net annual revenue. 80% of the firms are used for training with 5-fold cross-validation, 20% are set aside and touched once.
5. Compares the performance on train and on test.
6. Turns the measured error into a 95% range.
7. Applies both models to the matched licensees and writes the client file.
8. Part 5: counts how many firms in the whole Romanian register the model could estimate.

Only the quant route is used. The lookup in step 2 is there to find out which company a name is and to collect the model's inputs (age, industry, city, legal form). It never collects revenue or headcount.

In [1]:
import os, re, json, time, hashlib
import numpy as np
import pandas as pd

# ---------- files ----------
licensee_list  = r"C:\Users\Cristi\Downloads\licensee_list.csv"
firme          = r"C:\Users\Cristi\Downloads\od_firme.csv"
web_uu         = r"C:\Users\Cristi\Downloads\web_uu_an2024.txt"
caen_categorii = r"C:\Users\Cristi\Downloads\caen_categorii.csv"
stare_firma    = r"C:\Users\Cristi\Downloads\od_stare_firma.csv"       # only for Part 5
caen_autorizat = r"C:\Users\Cristi\Downloads\od_caen_autorizat.csv"    # only for Part 5
OUT_DIR        = r"C:\Users\Cristi\Downloads"

# ---------- matching ----------
COL_NAME    = None      # column with the company name in the client's list; None = found automatically
COL_COUNTRY = None      # column with the country; None = first column with "country" in its name
MIN_CONFIDENCE = 75     # keep a match only if the name similarity is at least this (0-100)
UK_API_KEY  = ""        # free key from Companies House; leave empty to skip UK names
USE_GLEIF   = False     # GLEIF is the official global LEI register, but it is not a site "from the country"
RO_SOURCE   = "https://data.gov.ro (ONRC open data, file od_firme.csv)"

# ---------- model ----------
REF_YEAR   = 2024       # the filings describe financial year 2024
N_FOLDS    = 5
TEST_SIZE  = 0.20
N_ITER     = 500
SEED       = 42
RON_PER_EUR = 4.9746    # average 2024 rate, used only to show revenue in EUR as well

In [2]:
def read_cols(path, usecols=None, **kw):
    """Read a text file without knowing the separator. Only the columns asked for are loaded."""
    with open(path, "rb") as f:
        first = f.readline().decode("latin-1")
    sep = max(["^", ";", "|", "\t", ","], key=first.count)
    header = [c.strip().strip('"').lstrip("\ufeff").lstrip("ï»¿").upper() for c in first.rstrip("\r\n").split(sep)]
    use = None
    if usecols is not None:
        missing = [c for c in usecols if c not in header]
        assert not missing, f"{path}: missing columns {missing}. Found: {header}"
        use = usecols
    for enc in ("utf-8-sig", "cp1250", "latin-1"):
        try:
            df = pd.read_csv(path, sep=sep, names=header, header=0, usecols=use, dtype=str,
                             encoding=enc, on_bad_lines="skip", **kw)
            break
        except UnicodeDecodeError:
            continue
    print(f"{os.path.basename(path)}: {len(df):,} rows, columns {list(df.columns)}")
    return df

def norm_cui(s):
    return (s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
             .str.upper().str.replace(r"^RO", "", regex=True).str.lstrip("0"))

LEGAL_WORDS = (r"\b(?:SRL D|SRL|SA|SC|SNC|SCS|PFA|II|IF|LTD|LIMITED|LLC|LLP|INC|CORP|CORPORATION|CO|COMPANY|GMBH|AG|KG|"
               r"BV|NV|SPA|SARL|SAS|SASU|EURL|PLC|OY|OYJ|AB|AS|ASA|APS|KFT|SRO|SP Z O O|DOO|HOLDING|HOLDINGS|GROUP)\b")

def ascii_upper(s):
    s = s.fillna("").astype("string").str.upper().str.normalize("NFKD")
    return s.str.encode("ascii", "ignore").str.decode("ascii")

def clean_name(s):
    """Same cleaning on both sides: upper case, no diacritics, no dots, no legal suffix, single spaces."""
    s = ascii_upper(s).str.replace(".", "", regex=False).str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    s = s.str.replace(LEGAL_WORDS, " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def clean_one(x):
    return clean_name(pd.Series([x])).iloc[0]

## Step 1. Read the client's list

I look at the list before doing anything with it: which column holds the name, which one holds the country, how many rows have no country at all. The country is turned into a two-letter code so that every row can be sent to the register of its own country.

In [3]:
lic = read_cols(licensee_list)
if COL_NAME is None:
    # a column called name/company/licensee (and not an id); if there is none, the column with the longest text
    named = [c for c in lic.columns if re.search(r"NAME|COMPANY|LICENSEE|DENUMIRE|FIRMA", c) and not re.search(r"(^|_)ID($|_)", c)]
    COL_NAME = named[0] if named else max(lic.columns, key=lambda c: lic[c].astype(str).str.len().mean())
print("First names read:", lic[COL_NAME].head(3).tolist(), "<- these must be company names, not codes")
if COL_COUNTRY is None:
    cand = [c for c in lic.columns if "COUNTRY" in c or "TARA" in c]
    COL_COUNTRY = cand[0] if cand else None
print("Name column:", COL_NAME, "| country column:", COL_COUNTRY)

ISO = {"ROMANIA": "RO", "FRANCE": "FR", "UNITED KINGDOM": "GB", "UK": "GB", "GREAT BRITAIN": "GB", "ENGLAND": "GB",
       "NORWAY": "NO", "FINLAND": "FI", "GERMANY": "DE", "NETHERLANDS": "NL", "THE NETHERLANDS": "NL",
       "UNITED STATES": "US", "USA": "US", "UNITED STATES OF AMERICA": "US", "ITALY": "IT", "SPAIN": "ES",
       "SWEDEN": "SE", "DENMARK": "DK", "BELGIUM": "BE", "AUSTRIA": "AT", "SWITZERLAND": "CH", "POLAND": "PL",
       "CZECHIA": "CZ", "CZECH REPUBLIC": "CZ", "HUNGARY": "HU", "IRELAND": "IE", "PORTUGAL": "PT",
       "BULGARIA": "BG", "GREECE": "GR", "CANADA": "CA", "AUSTRALIA": "AU", "JAPAN": "JP", "CHINA": "CN",
       "INDIA": "IN", "BRAZIL": "BR", "MEXICO": "MX", "TURKEY": "TR", "LUXEMBOURG": "LU"}

def to_iso(x):
    x = clean_one(x) if isinstance(x, str) else ""
    if len(x) == 2: return "GB" if x == "UK" else x
    return ISO.get(x, "")

lic["row_id"] = range(1, len(lic) + 1)
lic["client_name"] = lic[COL_NAME]
lic["country_iso"] = lic[COL_COUNTRY].map(to_iso) if COL_COUNTRY else ""
lic["name_clean"] = clean_name(lic["client_name"])
print("\nRows per country (empty = no country given or not recognised):")
print(lic["country_iso"].value_counts(dropna=False))
lic.head(10)

licensee_list.csv: 100 rows, columns ['ID', 'LICENSEE_NAME', 'ADDRESS', 'COUNTRY', 'WEBSITE']
First names read: ['Siemens S.A. de C.V.', 'SIEMENS HEALTHINEERS', 'Siemens Industry Software (Shanghai) Co., Ltd.'] <- these must be company names, not codes
Name column: LICENSEE_NAME | country column: COUNTRY

Rows per country (empty = no country given or not recognised):
country_iso
US    19
      14
GB    12
DE    12
RO     9
CH     5
FR     3
KR     2
CA     2
IN     2
SG     2
LU     2
SE     2
TR     1
CN     1
MX     1
ES     1
BR     1
NG     1
SK     1
IE     1
PK     1
NO     1
AU     1
FI     1
PT     1
NL     1
Name: count, dtype: int64


,ID,LICENSEE_NAME,ADDRESS,COUNTRY,WEBSITE,row_id,client_name,country_iso,name_clean
0,IN-001,Siemens S.A. de C.V.,NaN,MX,NaN,1,Siemens S.A. de C.V.,MX,SIEMENS DE CV
1,IN-002,SIEMENS HEALTHINEERS,NaN,NaN,NaN,2,SIEMENS HEALTHINEERS,,SIEMENS HEALTHINEERS
2,IN-003,"Siemens Industry Software (Shanghai) Co., Ltd.",Shanghai,CN,NaN,3,"Siemens Industry Software (Shanghai) Co., Ltd.",CN,SIEMENS INDUSTRY SOFTWARE SHANGHAI
3,IN-004,Siemens Ltd.,"Mumbai, Maharashtra",NaN,NaN,4,Siemens Ltd.,,SIEMENS
4,IN-005,SIEMENS MOBILITY LIMITED,NaN,GB,NaN,5,SIEMENS MOBILITY LIMITED,GB,SIEMENS MOBILITY
5,IN-006,Siemens Ltda.,NaN,BR,NaN,6,Siemens Ltda.,BR,SIEMENS LTDA
6,IN-007,Siemens Energy,NaN,NaN,NaN,7,Siemens Energy,,SIEMENS ENERGY
7,IN-008,"Siemens Gamesa Renewable Energy, S.A.",NaN,ES,NaN,8,"Siemens Gamesa Renewable Energy, S.A.",ES,SIEMENS GAMESA RENEWABLE ENERGY
8,IN-009,SIEMENS PTE. LTD.,NaN,SG,NaN,9,SIEMENS PTE. LTD.,SG,SIEMENS PTE
9,IN-010,Siemens Canada Limited,NaN,CA,NaN,10,Siemens Canada Limited,CA,SIEMENS CANADA


## Step 2. One official source per country

Each source below is the public register of that country (or its official open data file). A source takes a name and gives back candidates in one common shape: registered name, registration number, founding year, legal form, industry code, city, and the link where the record can be seen.

Romania: the ONRC open data file, read locally. France: the government company search (recherche-entreprises.api.gouv.fr). Norway: Brønnøysund register (data.brreg.no). Finland: PRH/YTJ open data. United Kingdom: Companies House (needs a free key). Any other country has no source here, and its rows are reported as "no official source for this country", which is my failure and not proof that the company does not exist.

All industry codes in these registers follow the same European NACE structure as the Romanian CAEN code, so the first four digits can be used directly.

In [4]:
import requests
from rapidfuzz import process, fuzz

CACHE = os.path.join(OUT_DIR, "registry_cache"); os.makedirs(CACHE, exist_ok=True)
HEADERS = {"User-Agent": "licensee-size-research/1.0", "Accept": "application/json"}

def get_json(url, params=None, auth=None):
    """One polite request, saved on disk so the same question is never asked twice."""
    key = hashlib.md5((url + json.dumps(params or {}, sort_keys=True)).encode()).hexdigest()
    path = os.path.join(CACHE, key + ".json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f: return json.load(f)
    r = requests.get(url, params=params, headers=HEADERS, auth=auth, timeout=20)
    time.sleep(1.0)
    r.raise_for_status()
    data = r.json()
    with open(path, "w", encoding="utf-8") as f: json.dump(data, f)
    return data

def year_of(x):
    m = re.search(r"(1[89]\d\d|20\d\d)", str(x or ""))
    return int(m.group(1)) if m else np.nan

def nace4(x):
    d = re.sub(r"\D", "", str(x or ""))
    return d[:4].zfill(4) if len(d) >= 3 else ""

def cand(name, reg_id, year, form, code, city, url):
    return {"matched_entity": name, "registration_id": str(reg_id), "founding_year": year,
            "legal_form_raw": form or "", "nace4": nace4(code), "city": city or "", "source_url": url}

# ---- Romania: the register file itself ----
df_firme = read_cols(firme, usecols=["DENUMIRE", "CUI", "COD_INMATRICULARE", "DATA_INMATRICULARE", "FORMA_JURIDICA", "ADR_JUDET"])
df_firme["CUI"] = norm_cui(df_firme["CUI"])
df_firme = df_firme[df_firme["CUI"].str.len() > 0].drop_duplicates("CUI")
df_firme["name_clean"] = clean_name(df_firme["DENUMIRE"])
ro_names = df_firme["name_clean"].drop_duplicates()
ro_names = ro_names[ro_names.str.len() > 2].tolist()
ro_by_name = df_firme.groupby("name_clean").indices

def src_ro(name):
    q = clean_one(name)
    hits = process.extract(q, ro_names, scorer=fuzz.token_sort_ratio, limit=3, score_cutoff=70)
    out = []
    for h in hits:
        for i in ro_by_name[h[0]][:5]:
            r = df_firme.iloc[i]
            out.append(cand(r["DENUMIRE"], r["CUI"], year_of(str(r["DATA_INMATRICULARE"])[:10]), r["FORMA_JURIDICA"],
                            "", r["ADR_JUDET"], f"{RO_SOURCE}, CUI {r['CUI']}"))
    return out

# ---- France ----
FR_FORM = {"54": "SARL", "57": "SAS", "55": "SA", "56": "SA", "52": "SNC", "53": "SCS", "10": "EI"}
def src_fr(name):
    d = get_json("https://recherche-entreprises.api.gouv.fr/search", {"q": name, "per_page": 5})
    out = []
    for r in d.get("results", []):
        siege = r.get("siege") or {}
        out.append(cand(r.get("nom_raison_sociale") or r.get("nom_complet"), r.get("siren"), year_of(r.get("date_creation")),
                        FR_FORM.get(str(r.get("nature_juridique") or "")[:2], ""), r.get("activite_principale"),
                        siege.get("libelle_commune"), f"https://annuaire-entreprises.data.gouv.fr/entreprise/{r.get('siren')}"))
    return out

# ---- Norway ----
def src_no(name):
    d = get_json("https://data.brreg.no/enhetsregisteret/api/enheter", {"navn": name, "size": 5})
    out = []
    for r in (d.get("_embedded") or {}).get("enheter", []):
        adr = r.get("forretningsadresse") or {}
        out.append(cand(r.get("navn"), r.get("organisasjonsnummer"),
                        year_of(r.get("stiftelsesdato") or r.get("registreringsdatoEnhetsregisteret")),
                        (r.get("organisasjonsform") or {}).get("kode"), (r.get("naeringskode1") or {}).get("kode"),
                        adr.get("kommune") or adr.get("poststed"),
                        f"https://data.brreg.no/enhetsregisteret/api/enheter/{r.get('organisasjonsnummer')}"))
    return out

# ---- Finland ----
def src_fi(name):
    d = get_json("https://avoindata.prh.fi/opendata-ytj-api/v3/companies", {"name": name})
    out = []
    for r in d.get("companies", [])[:5]:
        bid = (r.get("businessId") or {}).get("value")
        nm = next((n.get("name") for n in r.get("names", []) if not n.get("endDate")), None) or \
             next((n.get("name") for n in r.get("names", [])), None)
        form = " ".join(x.get("description", "") for f in r.get("companyForms", []) for x in f.get("descriptions", []))
        form = "OYJ" if "PUBLIC" in form.upper() or "JULKINEN" in form.upper() else ("OY" if "LIMITED" in form.upper() or "OSAKEYHTI" in form.upper() else form)
        city = next((p.get("city") for a in r.get("addresses", []) for p in a.get("postOffices", [])), "")
        out.append(cand(nm, bid, year_of(r.get("registrationDate")), form, (r.get("mainBusinessLine") or {}).get("type"),
                        city, f"https://avoindata.prh.fi/opendata-ytj-api/v3/companies?businessId={bid}"))
    return out

# ---- United Kingdom ----
def src_gb(name):
    base = "https://api.company-information.service.gov.uk"
    d = get_json(base + "/search/companies", {"q": name, "items_per_page": 5}, auth=(UK_API_KEY, ""))
    out = []
    for r in d.get("items", []):
        num = r.get("company_number")
        sic = ""
        if fuzz.token_sort_ratio(clean_one(name), clean_one(r.get("title", ""))) >= MIN_CONFIDENCE:
            sic = (get_json(f"{base}/company/{num}", auth=(UK_API_KEY, "")).get("sic_codes") or [""])[0]
        out.append(cand(r.get("title"), num, year_of(r.get("date_of_creation")), r.get("company_type"), sic,
                        (r.get("address") or {}).get("locality"),
                        f"https://find-and-update.company-information.service.gov.uk/company/{num}"))
    return out

# ---- GLEIF (off by default) ----
def src_gleif(name, iso):
    d = get_json("https://api.gleif.org/api/v1/lei-records",
                 {"filter[entity.legalName]": name, "filter[entity.legalAddress.country]": iso, "page[size]": 5})
    out = []
    for r in d.get("data", []):
        a = r.get("attributes") or {}; e = a.get("entity") or {}
        out.append(cand((e.get("legalName") or {}).get("name"), a.get("lei"), year_of(e.get("creationDate")),
                        (e.get("legalForm") or {}).get("other"), "", (e.get("legalAddress") or {}).get("city"),
                        f"https://search.gleif.org/#/record/{a.get('lei')}"))
    return out

SOURCES = {"RO": src_ro, "FR": src_fr, "NO": src_no, "FI": src_fi}
if UK_API_KEY: SOURCES["GB"] = src_gb
print("Countries with an official source:", sorted(SOURCES))

od_firme.csv: 4,116,148 rows, columns ['DENUMIRE', 'CUI', 'COD_INMATRICULARE', 'DATA_INMATRICULARE', 'FORMA_JURIDICA', 'ADR_JUDET']
Countries with an official source: ['FI', 'FR', 'NO', 'RO']


## Step 3. Match, and keep only what I am 75% sure about

For every row I ask the source of its country for candidates and compare the cleaned client name with each cleaned candidate name. The result is a similarity from 0 to 100. This is a score, not a true probability, but it is the number the 75% rule is applied to.

A row is kept as matched only if the best candidate scores at least 75 and no second, different company also scores at least 75. One exception: if exactly one company has the identical cleaned name (score 100), it is kept even when similar names exist. Everything else goes into a bucket that says why:

- matched: one company, score at least 75.
- ambiguous: two or more different companies score at least 75. I failed to choose.
- low confidence: the source answered, but the best name is below 75. Often a brand or an abbreviation. I failed.
- not found in the register: the source answered and had no candidate at all. This is the only bucket that hints the company may not exist under that name in that country.
- no official source for this country, no country given, source error: all three are my failure, not a statement about the company.

In [5]:
rows = []
for _, r in lic.iterrows():
    rec = {"row_id": r["row_id"], "client_name": r["client_name"], "country_iso": r["country_iso"],
           "bucket": None, "match_confidence": np.nan, "match_note": ""}
    iso, name = r["country_iso"], str(r["client_name"])
    if len(r["name_clean"]) < 3:
        rec["bucket"] = "name_too_short"
    elif not iso:
        rec["bucket"] = "no_country_given"
    elif iso not in SOURCES and not USE_GLEIF:
        rec["bucket"] = "no_official_source_for_country"
    else:
        try:
            cands = SOURCES[iso](name) if iso in SOURCES else src_gleif(name, iso)
            for c in cands:
                c["score"] = fuzz.token_sort_ratio(r["name_clean"], clean_one(c["matched_entity"] or ""))
            cands = sorted(cands, key=lambda c: -c["score"])
            good = {c["registration_id"] for c in cands if c["score"] >= MIN_CONFIDENCE}
            top = {c["registration_id"] for c in cands if c["score"] == cands[0]["score"]} if cands else set()
            if len(top) == 1 and cands[0]["score"] == 100:
                good = top                       # one company with exactly this name: near-misses do not make it ambiguous
            if not cands:
                rec["bucket"] = "not_found_in_register"
            elif len(good) == 1:
                rec.update(cands[0]); rec["bucket"] = "matched"; rec["match_confidence"] = cands[0]["score"]
            elif len(good) > 1:
                rec["bucket"] = "ambiguous_several_companies"; rec["match_confidence"] = cands[0]["score"]
                rec["match_note"] = f"{len(good)} different companies score >= {MIN_CONFIDENCE}"
            else:
                rec["bucket"] = "low_confidence"; rec["match_confidence"] = cands[0]["score"]
                rec["match_note"] = f"closest: {cands[0]['matched_entity']} ({cands[0]['source_url']})"
        except Exception as e:
            rec["bucket"] = "source_error"; rec["match_note"] = f"{type(e).__name__}: {str(e)[:120]}"
    rows.append(rec)

match = pd.DataFrame(rows).drop(columns=["score"], errors="ignore")
for c in ["matched_entity", "registration_id", "founding_year", "legal_form_raw", "nace4", "city", "source_url"]:
    if c not in match.columns: match[c] = np.nan
print(match["bucket"].value_counts())
matched = match[match["bucket"] == "matched"].copy()
print(f"\nKept: {len(matched)} of {len(match)} names")
match.to_csv(os.path.join(OUT_DIR, "match_table.csv"), index=False, encoding="utf-8-sig")
match.head(20)

bucket
no_official_source_for_country    71
no_country_given                  14
matched                            9
ambiguous_several_companies        4
name_too_short                     1
not_found_in_register              1
Name: count, dtype: int64

Kept: 9 of 100 names


,row_id,client_name,country_iso,bucket,match_confidence,match_note,matched_entity,registration_id,founding_year,legal_form_raw,nace4,city,source_url
0,1,Siemens S.A. de C.V.,MX,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,SIEMENS HEALTHINEERS,,no_country_given,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,"Siemens Industry Software (Shanghai) Co., Ltd.",CN,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Siemens Ltd.,,no_country_given,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,SIEMENS MOBILITY LIMITED,GB,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Siemens Ltda.,BR,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Siemens Energy,,no_country_given,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,"Siemens Gamesa Renewable Energy, S.A.",ES,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,SIEMENS PTE. LTD.,SG,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,Siemens Canada Limited,CA,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 4. Training table from the Romanian filings

Two files, joined on the fiscal code (CUI): what the firm filed for 2024 (I13 = net annual revenue, I20 = average number of employees, CAEN = its main activity) and who the firm is (registration date, county, legal form). Only the needed columns are loaded. One firm is one row.

Before any model sees the table I check what kind of rows are in it:

- firms with negative revenue are removed, because the target is put on a log scale;
- firms with zero revenue and zero employees are removed: they filed, but nothing happened in them in 2024, and the client's licensees are operating businesses;
- the client's own matched firms are removed, so later they are firms the model has never seen.

In [6]:
df_uu = read_cols(web_uu, usecols=["CUI", "CAEN", "I13", "I20"])
df_uu["CUI"] = norm_cui(df_uu["CUI"])
df_uu["I13"] = pd.to_numeric(df_uu["I13"], errors="coerce")
df_uu["I20"] = pd.to_numeric(df_uu["I20"], errors="coerce")
df_uu = df_uu.drop_duplicates("CUI")

df = df_uu.merge(df_firme, on="CUI", how="inner")
print(f"Filings: {len(df_uu):,} | found in the register: {len(df):,}")

n0 = len(df)
df = df.dropna(subset=["I13", "I20"])
neg = (df["I13"] < 0) | (df["I20"] < 0)
dormant = (df["I13"] == 0) & (df["I20"] == 0)
client_cui = set(matched.loc[matched["country_iso"] == "RO", "registration_id"].astype(str))
is_client = df["CUI"].isin(client_cui)
print(f"Missing target: {n0 - len(df):,} | negative: {neg.sum():,} | zero revenue and zero employees: {dormant.sum():,} "
      f"| client's firms: {is_client.sum():,}")
print(f"Zero revenue only: {((df['I13'] == 0) & ~dormant).sum():,} | zero employees only: {((df['I20'] == 0) & ~dormant).sum():,} (kept)")

client_reported = df[is_client][["CUI", "I13", "I20", "CAEN"]].copy()
df = df[~neg & ~dormant & ~is_client].reset_index(drop=True)
print(f"Rows for modelling: {len(df):,}")

web_uu_an2024.txt: 822,292 rows, columns ['CUI', 'CAEN', 'I13', 'I20']
Filings: 822,292 | found in the register: 821,695
Missing target: 323,272 | negative: 176 | zero revenue and zero employees: 12,292 | client's firms: 0
Zero revenue only: 2,104 | zero employees only: 22,644 (kept)
Rows for modelling: 485,955


## Step 5. Features

Every feature has to exist both for a Romanian firm and for a foreign firm found in its own register, otherwise the model could not be applied to the client's list.

- firm_age: 2024 minus the registration year. Older firms had more time to grow.
- principal_field: the big category of the main activity code, from caen_categorii.csv.
- nace_division: the first two digits of the activity code, a finer industry.
- capital_city_flag: 1 if the firm is registered in the capital of its country.
- legal_form_group: private limited (SRL, Ltd, GmbH, SARL, AS, Oy...), joint stock (SA, PLC, AG, ASA, Oyj...), sole trader (PFA, II...), partnership, or other. A franchise is a contract, not a legal form, so no register records it and it cannot be a feature.

Not used on purpose: the other balance sheet lines (assets, profit). They predict revenue well, but a firm that has them has also reported its revenue, so there would be nothing left to estimate.

In [7]:
m = pd.read_csv(caen_categorii, dtype=str)
m.columns = m.columns.str.strip().str.upper()
m4 = m[m["NIVEL"] == "clasa"].drop_duplicates("COD_CAEN").set_index("COD_CAEN")["CATEGORIE_MARE"]
m2 = m[m["NIVEL"] == "divizie"].drop_duplicates("COD_CAEN").set_index("COD_CAEN")["CATEGORIE_MARE"]

CAPITALS = {"RO": ["BUCURESTI"], "FR": ["PARIS"], "GB": ["LONDON"], "NO": ["OSLO"], "FI": ["HELSINKI", "HELSINGFORS"],
            "DE": ["BERLIN"], "NL": ["AMSTERDAM"], "US": ["WASHINGTON"], "IT": ["ROMA", "ROME"], "ES": ["MADRID"],
            "SE": ["STOCKHOLM"], "DK": ["KOBENHAVN", "COPENHAGEN"], "BE": ["BRUXELLES", "BRUSSEL", "BRUSSELS"],
            "AT": ["WIEN", "VIENNA"], "CH": ["BERN"], "PL": ["WARSZAWA", "WARSAW"], "CZ": ["PRAHA", "PRAGUE"],
            "HU": ["BUDAPEST"], "IE": ["DUBLIN"], "PT": ["LISBOA", "LISBON"], "BG": ["SOFIA"], "GR": ["ATHENS", "ATHINA"]}

G_SOLE    = r"\b(?:PFA|II|IF|EI|ENK|SOLE TRADER|PERSOANA FIZICA|INTREPRINDERE INDIVIDUALA|INTREPRINDERE FAMILIALA|TOIMINIMI)\b"
G_JOINT   = r"\b(?:SA|PLC|AG|NV|ASA|OYJ|SPA|SE|INC|CORP|CORPORATION|PUBLIC LIMITED COMPANY)\b"
G_PRIVATE = r"\b(?:SRL D|SRL|LTD|LIMITED|LLC|GMBH|BV|SARL|SAS|SASU|EURL|AS|OY|AB|APS|SP Z O O|KFT|SRO|DOO|PRIVATE LIMITED COMPANY)\b"
G_PARTNER = r"\b(?:SNC|SCS|LLP|LP|KG|OHG|ANS|DA|KY)\b"

def legal_group(form, name):
    """Legal form group from the register's own field; if that says nothing, from the last words of the name."""
    def norm(x): return ascii_upper(x).str.replace(".", "", regex=False).str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    f = norm(form)
    tail = norm(name).str.split().str[-3:].str.join(" ")
    out = pd.Series("other", index=f.index)
    for text in (tail, f):                       # the register field is applied last, so it wins
        for label, pat in (("partnership", G_PARTNER), ("private_limited", G_PRIVATE), ("joint_stock", G_JOINT), ("sole_trader", G_SOLE)):
            out = out.mask(text.str.contains(pat, regex=True), label)
    return out

def build_features(founding_year, code4, city, country_iso, form, name):
    code4 = code4.fillna("").astype(str)
    code4 = code4.where(code4.str.fullmatch(r"\d{4}") & (code4 != "0000"))
    city_a = ascii_upper(city)
    cap = pd.Series(0, index=city_a.index, dtype="int8")
    for iso, names in CAPITALS.items():
        cap = cap.mask((country_iso == iso) & city_a.str.contains("|".join(names), regex=True), 1)
    age = REF_YEAR - pd.to_numeric(founding_year, errors="coerce")
    return pd.DataFrame({
        "firm_age": age.where(age >= 0),
        "principal_field": code4.map(m4).fillna(code4.str[:2].map(m2)),
        "nace_division": code4.str[:2],
        "capital_city_flag": cap,
        "legal_form_group": legal_group(form, name),
    })

CAT_COLS = ["principal_field", "nace_division", "legal_form_group"]
NUM_COLS = ["firm_age", "capital_city_flag"]
FEATURES = CAT_COLS + NUM_COLS

X_all = build_features(
    founding_year=pd.to_datetime(df["DATA_INMATRICULARE"].astype("string").str[:10], format="%d/%m/%Y", errors="coerce").dt.year,
    code4=df["CAEN"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(4).str[:4],
    city=df["ADR_JUDET"], country_iso=pd.Series("RO", index=df.index), form=df["FORMA_JURIDICA"], name=df["DENUMIRE"])

def prep(X):
    X = X[FEATURES].copy()
    for c in CAT_COLS: X[c] = X[c].astype("string").fillna("missing").astype(str)
    for c in NUM_COLS: X[c] = pd.to_numeric(X[c], errors="coerce")
    return X

X_all = prep(X_all)
y_all = {"employees": np.log1p(df["I20"]), "revenue": np.log1p(df["I13"])}      # both targets on the log(1 + x) scale
print("Missing values per feature:"); print((X_all == "missing").sum()[CAT_COLS].to_string(), "\nfirm_age", X_all["firm_age"].isna().sum())
print("\nLegal form groups:"); print(X_all["legal_form_group"].value_counts())
X_all.head()

Missing values per feature:
principal_field     0
nace_division       0
legal_form_group    0 
firm_age 1

Legal form groups:
legal_form_group
private_limited    482792
other                1451
joint_stock          1370
partnership           335
sole_trader             7
Name: count, dtype: int64


,principal_field,nace_division,legal_form_group,firm_age,capital_city_flag
0,"Agricultură, silvicultură, pescuit",01,private_limited,23.0,0
1,"Agricultură, silvicultură, pescuit",01,private_limited,0.0,0
2,"Agricultură, silvicultură, pescuit",01,private_limited,11.0,0
3,"Agricultură, silvicultură, pescuit",01,private_limited,1.0,0
4,"Agricultură, silvicultură, pescuit",01,private_limited,4.0,1


## Step 6. Split 80/20, then 5-fold cross-validation on the 80%

First the split: 20% of the firms are put aside and are not used for anything until the final check.

Then, inside the 80%, the firms are cut into 5 equal parts. The model is trained 5 times, each time on 4 parts, and predicts the part it did not see. This gives, for every training firm, one prediction made by a model that never saw that firm. It also shows whether the result is stable or depends on which firms happened to land in the training set.

At the end one final model per target is trained on the whole 80%.

Both targets are right-skewed (many small firms, a few huge ones), so the models learn log(1 + x) and the predictions are turned back with exp(x) - 1.

In [8]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split, KFold

idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED)
print(f"train: {len(idx_train):,} firms | test (set aside): {len(idx_test):,} firms")

def new_model():
    return CatBoostRegressor(loss_function="RMSE", iterations=N_ITER, learning_rate=0.1, depth=6,
                             random_seed=SEED, verbose=False)

def pool(i, y=None):
    return Pool(X_all.iloc[i], None if y is None else y.iloc[i], cat_features=CAT_COLS)

models, oof, fold_scores = {}, {}, []
for target, y in y_all.items():
    oof[target] = pd.Series(np.nan, index=idx_train)
    for k, (a, b) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(idx_train), start=1):
        tr, va = idx_train[a], idx_train[b]
        mdl = new_model().fit(pool(tr, y))
        p = mdl.predict(pool(va))
        oof[target].loc[va] = p
        err = np.abs(y.iloc[va].to_numpy() - p)
        fold_scores.append({"target": target, "fold": k, "MAE_log": err.mean(),
                            "R2_log": 1 - ((y.iloc[va] - p) ** 2).sum() / ((y.iloc[va] - y.iloc[va].mean()) ** 2).sum()})
        print(f"{target} fold {k}: MAE_log {fold_scores[-1]['MAE_log']:.3f}  R2_log {fold_scores[-1]['R2_log']:.3f}")
    models[target] = new_model().fit(pool(idx_train, y))

fold_scores = pd.DataFrame(fold_scores)
fold_scores.groupby("target")[["MAE_log", "R2_log"]].agg(["mean", "std"]).round(4)

train: 388,764 firms | test (set aside): 97,191 firms
employees fold 1: MAE_log 0.405  R2_log 0.116
employees fold 2: MAE_log 0.406  R2_log 0.122
employees fold 3: MAE_log 0.407  R2_log 0.118
employees fold 4: MAE_log 0.405  R2_log 0.117
employees fold 5: MAE_log 0.407  R2_log 0.115
revenue fold 1: MAE_log 1.158  R2_log 0.078
revenue fold 2: MAE_log 1.153  R2_log 0.083
revenue fold 3: MAE_log 1.157  R2_log 0.080
revenue fold 4: MAE_log 1.160  R2_log 0.080
revenue fold 5: MAE_log 1.158  R2_log 0.081


MAE_log          R2_log        
             mean     std    mean     std
target                                   
employees  0.4060  0.0010  0.1176  0.0028
revenue    1.1569  0.0027  0.0802  0.0016

## Step 7. Performance: train against test

Three rows per target:

- train: the final model on the firms it learned from. Always the most flattering number.
- cv: the predictions from the 5 folds, each on firms that fold's model did not see.
- test: the final model on the 20% set aside. This is the number that counts.

If train is much better than test, the model memorised its firms. If cv and test agree, the result is stable.

How to read the columns: R2_log is the share of the differences between firms that the model explains (1 is perfect, 0 is no better than a constant). MAE_log_baseline is the error of the laziest possible model, which gives every firm the median. Median error factor 2 means the typical prediction is 2 times too high or 2 times too low. The last column is the share of firms predicted within a factor of 2.

In [9]:
def score(real, pred, baseline):
    err = np.abs(real - pred)
    return {"R2_log": 1 - ((real - pred) ** 2).sum() / ((real - real.mean()) ** 2).sum(),
            "MAE_log": err.mean(), "MAE_log_baseline": np.abs(real - baseline).mean(),
            "median_error_factor": np.exp(np.median(err)), "pct_within_factor_2": (err <= np.log(2)).mean() * 100}

perf, test_resid = [], {}
for target, y in y_all.items():
    base = y.iloc[idx_train].median()
    p_train, p_test = models[target].predict(pool(idx_train)), models[target].predict(pool(idx_test))
    test_resid[target] = y.iloc[idx_test].to_numpy() - p_test
    perf.append({"target": target, "set": "train", **score(y.iloc[idx_train].to_numpy(), p_train, base)})
    perf.append({"target": target, "set": "cv",    **score(y.iloc[idx_train].to_numpy(), oof[target].to_numpy(), base)})
    perf.append({"target": target, "set": "test",  **score(y.iloc[idx_test].to_numpy(), p_test, base)})

perf = pd.DataFrame(perf).set_index(["target", "set"]).round(3)
importance = pd.DataFrame({t: pd.Series(models[t].get_feature_importance(), index=FEATURES) for t in models}).round(1)
perf.to_csv(os.path.join(OUT_DIR, "performance_train_vs_test.csv"), encoding="utf-8-sig")
importance.to_csv(os.path.join(OUT_DIR, "feature_importance.csv"), encoding="utf-8-sig")
display(perf); display(importance)

R2_log  MAE_log  MAE_log_baseline  median_error_factor  \
target    set                                                             
employees train   0.121    0.405             0.396                1.340   
          cv      0.118    0.406             0.396                1.343   
          test    0.118    0.406             0.398                1.340   
revenue   train   0.083    1.155             1.210                2.378   
          cv      0.080    1.157             1.210                2.381   
          test    0.081    1.161             1.215                2.381   

                 pct_within_factor_2  
target    set                         
employees train               83.339  
          cv                  83.266  
          test                83.176  
revenue   train               41.797  
          cv                  41.742  
          test                41.574

,employees,revenue
principal_field,38.7,29.3
nace_division,26.1,13.3
legal_form_group,2.5,2.9
firm_age,29.9,49.0
capital_city_flag,2.7,5.4


## Step 8. From measured error to a 95% range

The brief asks for a range whose width comes from the error measured on the firms set aside. So I take the errors on the test set (real minus predicted, on the log scale) and keep the 2.5th and the 97.5th percentile. A firm's range is its prediction plus these two numbers, turned back to plain units.

Strictly speaking this is a prediction interval (where one firm's true value should fall), not a confidence interval (which is about an average). It is the one the client needs.

Check: the same two percentiles computed from the cross-validation errors should be close to the test ones. I also apply the cross-validation range to the test firms and count how many fall inside; it should be near 95%.

In [10]:
LOW, HIGH = 0.025, 0.975
ranges, check = {}, []
for target, y in y_all.items():
    q_test = np.quantile(test_resid[target], [LOW, HIGH])
    q_cv = np.quantile(y.iloc[idx_train].to_numpy() - oof[target].to_numpy(), [LOW, HIGH])
    ranges[target] = q_test
    cover = ((test_resid[target] >= q_cv[0]) & (test_resid[target] <= q_cv[1])).mean() * 100
    check.append({"target": target, "low_factor_test": np.exp(q_test[0]), "high_factor_test": np.exp(q_test[1]),
                  "low_factor_cv": np.exp(q_cv[0]), "high_factor_cv": np.exp(q_cv[1]),
                  "pct_of_test_firms_inside_cv_range": cover})
range_check = pd.DataFrame(check).set_index("target").round(3)
range_check.to_csv(os.path.join(OUT_DIR, "range_check.csv"), encoding="utf-8-sig")
range_check     # low/high factor: the true value is expected between prediction x low and prediction x high

,low_factor_test,high_factor_test,low_factor_cv,high_factor_cv,pct_of_test_firms_inside_cv_range
target,,,,,
employees,0.389,4.020,0.396,4.055,94.876
revenue,0.027,13.992,0.029,13.762,94.827


## Step 9. Apply both models to the matched licensees

The matched firms get the same features, built by the same function. For Romanian firms the industry code comes from their filing, for the others from their own register.

Every row of the client file carries: the name as the client wrote it, the bucket, the match confidence, the exact entity and its registration number, the link to the source, the inputs the model used, the estimate with its 95% range for both numbers, and the reported 2024 figure where the firm filed one. When a reported figure exists it is the better number and the estimate next to it is only a check on the model.

Revenue is in RON for financial year 2024 because the model learned from Romanian filings; a EUR column is added at the average 2024 rate.

In [11]:
out = match.copy()
if len(matched):
    mt = matched.copy()
    rep = client_reported.set_index("CUI")
    is_ro = mt["country_iso"] == "RO"
    mt.loc[is_ro, "nace4"] = mt.loc[is_ro, "registration_id"].map(rep["CAEN"]).astype("string").str.replace(r"\D", "", regex=True).str.zfill(4)
    Xm = prep(build_features(mt["founding_year"], mt["nace4"], mt["city"], mt["country_iso"], mt["legal_form_raw"], mt["matched_entity"]))
    res = Xm.add_prefix("input_")
    for target, short in (("employees", "employees"), ("revenue", "revenue_ron")):
        p = models[target].predict(Pool(Xm, cat_features=CAT_COLS))
        res[f"{short}_estimate"] = np.expm1(p).clip(min=0).round()
        res[f"{short}_low95"] = np.expm1(p + ranges[target][0]).clip(min=0).round()
        res[f"{short}_high95"] = np.expm1(p + ranges[target][1]).clip(min=0).round()
    for c in ("estimate", "low95", "high95"):
        res[f"revenue_eur_{c}"] = (res[f"revenue_ron_{c}"] / RON_PER_EUR).round()
    res["employees_reported_2024"] = mt["registration_id"].map(rep["I20"]).where(is_ro)
    res["revenue_ron_reported_2024"] = mt["registration_id"].map(rep["I13"]).where(is_ro)
    res["inputs_missing"] = (Xm[CAT_COLS] == "missing").sum(axis=1) + Xm["firm_age"].isna().astype(int)
    out = out.join(res)

    v = out.dropna(subset=["employees_reported_2024"]) if "employees_reported_2024" in out else out.iloc[:0]
    if len(v):
        print(f"Matched firms with a reported 2024 figure: {len(v)}")
        print(f"  employees: reported value inside the 95% range for {v['employees_reported_2024'].between(v['employees_low95'], v['employees_high95']).mean():.0%}")
        print(f"  revenue:   reported value inside the 95% range for {v['revenue_ron_reported_2024'].between(v['revenue_ron_low95'], v['revenue_ron_high95']).mean():.0%}")
    else:
        print("No matched firm has a reported figure, so the model cannot be checked on the client's own firms.")
else:
    print("No name passed the 75% rule, so there is nothing to estimate.")

out["period"] = f"financial year {REF_YEAR}"
out["method"] = "CatBoost trained on Romanian 2024 filings; 95% range from the error on firms set aside"
out = out.drop(columns=["founding_year", "legal_form_raw", "nace4", "city"], errors="ignore")
out.to_csv(os.path.join(OUT_DIR, "client_file.csv"), index=False, encoding="utf-8-sig")
out.to_excel(os.path.join(OUT_DIR, "client_file.xlsx"), index=False)
out

No matched firm has a reported figure, so the model cannot be checked on the client's own firms.


,row_id,client_name,country_iso,bucket,match_confidence,match_note,matched_entity,registration_id,source_url,input_principal_field,...,revenue_ron_low95,revenue_ron_high95,revenue_eur_estimate,revenue_eur_low95,revenue_eur_high95,employees_reported_2024,revenue_ron_reported_2024,inputs_missing,period,method
0,1,Siemens S.A. de C.V.,MX,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
1,2,SIEMENS HEALTHINEERS,,no_country_given,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
2,3,"Siemens Industry Software (Shanghai) Co., Ltd.",CN,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
3,4,Siemens Ltd.,,no_country_given,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
4,5,SIEMENS MOBILITY LIMITED,GB,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,"Southern Glazer's Wine & Spirits, LLC",US,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
96,97,"Reyes Holdings, L.L.C.",US,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
97,98,"Prime Healthcare Services, Inc.",US,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...
98,99,"Drury Hotels Company, LLC",US,no_official_source_for_country,NaN,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financial year 2024,CatBoost trained on Romanian 2024 filings; 95%...


## Part 5. On how many firms in the Romanian register could the model produce an estimate?

Living or not: the last status row of each firm in od_stare_firma, kept if the code is in ACTIVE_CODES (check the codes in n_stare_firma.csv; the table printed below shows how many firms sit on each code).

The line: a living firm for which all the model's inputs exist, meaning a registration date, a legal form, a county and an activity code that maps to a category. The activity code is taken from the 2024 filing; for firms that did not file, from the first code in od_caen_autorizat. The line is rough: it says the model can return a number, not that the number is good.

In [12]:
ACTIVE_CODES = ["1048"]

st = read_cols(stare_firma)
col_st = [c for c in st.columns if c != "COD_INMATRICULARE"][0]
st = st.drop_duplicates("COD_INMATRICULARE", keep="last")
reg = df_firme.merge(st, on="COD_INMATRICULARE", how="left")
print("\nFirms per status code (top 15):"); print(reg[col_st].value_counts(dropna=False).head(15))
alive = reg[reg[col_st].astype("string").str.strip().isin(ACTIVE_CODES)].copy()

alive["code"] = alive["CUI"].map(df_uu.set_index("CUI")["CAEN"])
if os.path.exists(caen_autorizat):
    first = []
    for chunk in pd.read_csv(caen_autorizat, sep="^", dtype=str, usecols=[0, 1], chunksize=2_000_000):
        chunk.columns = ["COD_INMATRICULARE", "CODE"]
        first.append(chunk.drop_duplicates("COD_INMATRICULARE"))
    first = pd.concat(first).drop_duplicates("COD_INMATRICULARE").set_index("COD_INMATRICULARE")["CODE"]
    alive["code"] = alive["code"].fillna(alive["COD_INMATRICULARE"].map(first))

f5 = build_features(pd.to_datetime(alive["DATA_INMATRICULARE"].astype("string").str[:10], format="%d/%m/%Y", errors="coerce").dt.year,
                    alive["code"].astype("string").str.replace(r"\D", "", regex=True).str.zfill(4).str[:4],
                    alive["ADR_JUDET"], pd.Series("RO", index=alive.index), alive["FORMA_JURIDICA"], alive["DENUMIRE"])
inside = f5[f5["firm_age"].notna() & f5["principal_field"].notna() & alive["FORMA_JURIDICA"].notna() & alive["ADR_JUDET"].notna()]

print(f"\nRegister, all firms with a CUI:  {len(reg):,}")
print(f"Register as filtered (living):   {len(alive):,}")
print(f"1) Inside the line:              {len(inside):,}")
print(f"2) Share of filtered register:   {len(inside) / max(len(alive), 1):.1%}")
print("3) Three largest industries:"); print(inside["principal_field"].value_counts().head(3))

od_stare_firma.csv: 4,564,268 rows, columns ['COD_INMATRICULARE', 'COD']

Firms per status code (top 15):
COD
1084    1958759
1048    1620179
2069     109633
1074     101452
1049      43340
2120      21596
2065      10722
1139       8554
1052       8029
1070       6141
2080       3833
2038       3504
1113       3250
2114       3030
1107       2243
Name: count, dtype: int64

Register, all firms with a CUI:  3,924,396
Register as filtered (living):   1,620,179
1) Inside the line:              1,433,312
2) Share of filtered register:   88.5%
3) Three largest industries:
principal_field
Comerț cu amănuntul                   204122
Construcții                           154295
Agricultură, silvicultură, pescuit    137569
Name: count, dtype: int64
